# Create and solve dense original economic dispatch problem
This notebook constructs and solves the full‑resolution (8760 hourly) economic dispatch model.  
It serves as the **baseline** for:
- computational performance (time & memory)
- objective value (total system cost)

**Author:** Zhi Gao  
**Affiliation:** Utrecht University  
**Date:** June 2026  

**License:** MIT  
This notebook is provided for academic reproducibility only.  
Please cite the associated publication if you use this code.

## Table of Contents
1. Scenario configuration
1. Import pacakages & data loading
1. Optimization model construction
1. Solve and save metrics

In [ ]:
# ============================================================
# Scenario Definition
# ============================================================
YEAR = 1985 # or 2014

# Structural assumptions
WITH_STORAGE = True
WITH_TRANSMISSION = True

# Operational extensions
WITH_GRID_TARIFF = False
WITH_STORAGE_DISSIPATION = False

print(f"Scenario: Year={YEAR}, "
      f"Storage={WITH_STORAGE}, "
      f"Transmission={WITH_TRANSMISSION}, "
      f"Tariff={WITH_GRID_TARIFF}, "
      f"Dissipation={WITH_STORAGE_DISSIPATION}")

In [ ]:
# ============================================================
# Import packages
# ============================================================

# Optimization
import pyomo.environ as pyo

# Data handling
import pandas as pd
import numpy as np

# System & timing
import timeit
import os

# Utilities
from utils.solver_utils import solve_with_gurobi_logging

# ============================================================
# Data Loading and Preprocessing
# ============================================================

# -------- Technology capacity --------
df_tech_capacity = pd.read_csv(
    "data/tech_capacity.csv",
    header=0, index_col=0
)

df_line_capacity = pd.read_csv(
    "data/line_capacity.csv",
    header=0, index_col=0
)

# -------- Apply scenario flags --------
if not WITH_STORAGE:
    df_tech_capacity["Battery"] *= 0
    df_tech_capacity["Battery_MWh"] *= 0
    df_tech_capacity["Hydro_Storage"] *= 0
    df_tech_capacity["Hydro_Storage_GWh"] *= 0
    
if not WITH_TRANSMISSION:
    df_line_capacity *= 0

# -------- Grid tariff --------
if WITH_GRID_TARIFF:
    df_grid_tarif = pd.read_csv(
        "data/grid_tarif.csv",
        index_col=0
    )

# ============================================================
# Technology Parameters
# ============================================================

# Storage characteristics
if WITH_STORAGE_DISSIPATION:
    dict_storage_hourly_self_dissipate_rate = {
        "Battery": 0.0001,
        "Hydro_Storage": 0.0001
    }

dict_storage_cycle_efficiency = {
    "Battery": 0.955,
    "Hydro_Storage": 0.875,
}

# Conversion efficiency
dict_conversion_efficiency = {
    "PEMEC": 0.585,
    "PEMFC": 0.5,
}

# Variable cost (Euro/MWh)
dict_variable_cost = {
    "Wind_Onshore": 2.06,
    "Wind_Offshore": 4.17,
    "Solar": 0.0,
    "CCGT": 79.0,
    "Battery": 0.1,
    "OCGT": 109.93,
    "Coal": 51.0,
    "Nuclear": 11.5,
    "E_ENS": 1000.0,
    "Run_of_River": 0.2,
    "Hydro_Storage": 0.2,
}

# ============================================================
# Sets Construction
# ============================================================

# Countries
list_country = np.unique(
    [asset[0:2] for asset in df_tech_capacity.index.values]
)

# Lines with positive capacity
list_line = [
    line for line in df_line_capacity.index.values
    if df_line_capacity.loc[line].values[0] > 0
]

# Country–technology output pairs
list_country_tech_output = []
for country in list_country:
    list_country_tech_output.append((country, "E_ENS"))
    for tech in [
        "Solar", "Wind_Offshore", "Wind_Onshore",
        "Nuclear", "Battery", "Coal", "CCGT",
        "OCGT", "Run_of_River", "Hydro_Storage"
    ]:
        if df_tech_capacity.loc[country, tech] > 0:
            list_country_tech_output.append((country, tech))

# Country–technology input (charging) pairs
list_country_tech_input = []
for country in list_country:
    for tech in ["Battery", "Hydro_Storage"]:
        if df_tech_capacity.loc[country, tech] > 0:
            list_country_tech_input.append((country, tech))

# ============================================================
# Time Series Profiles
# ============================================================

demand_profile = pd.read_csv(
    f"data/profiles/demand_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

solar_profile = pd.read_csv(
    f"data/profiles/solar_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

wind_offshore_profile = pd.read_csv(
    f"data/profiles/offshore_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

wind_onshore_profile = pd.read_csv(
    f"data/profiles/onshore_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

run_of_river_profile = pd.read_csv(
    f"data/profiles/run_of_river_profile_{YEAR}.csv",
    header=0, index_col=0
).round()

dict_tech_profile = {
    "Solar": solar_profile,
    "Wind_Offshore": wind_offshore_profile,
    "Wind_Onshore": wind_onshore_profile,
    "Run_of_River": run_of_river_profile,
}

print("✅ Data and sets successfully prepared.")

In [ ]:
# ============================================================
# Dense Model Construction (8760h)
# ============================================================

start_time = timeit.default_timer()

# -------- Sets --------
model_dense = pyo.ConcreteModel()

model_dense.set_country = pyo.Set(initialize=list_country)
model_dense.set_line = pyo.Set(initialize=list_line)
model_dense.set_hour = pyo.Set(initialize=range(1, 8761))
model_dense.set_country_tech_output = pyo.Set(
    initialize=list_country_tech_output
)
model_dense.set_country_tech_input = pyo.Set(
    initialize=list_country_tech_input
)

print(f"✅ Sets created in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# -------- Variables --------
model_dense.var_transmission = pyo.Var(
    model_dense.set_line,
    model_dense.set_hour,
    within=pyo.NonNegativeReals,
)

model_dense.var_tech_output = pyo.Var(
    model_dense.set_country_tech_output,
    model_dense.set_hour,
    within=pyo.NonNegativeReals,
)

model_dense.var_tech_input = pyo.Var(
    model_dense.set_country_tech_input,
    model_dense.set_hour,
    within=pyo.NonNegativeReals,
)

model_dense.var_stored_energy = pyo.Var(
    model_dense.set_country_tech_input,
    model_dense.set_hour,
    within=pyo.NonNegativeReals,
)

print(f"✅ Variables created in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# ============================================================
# Constraints
# ============================================================

# -------- Energy balance --------
def energy_balance_rule(m, country, hour):
    return (
        0.9 * pyo.quicksum(
            m.var_transmission[line, hour]
            for line in m.set_line
            if line[3:5] == country
        )
        - pyo.quicksum(
            m.var_transmission[line, hour]
            for line in m.set_line
            if line[0:2] == country
        )
        + pyo.quicksum(
            m.var_tech_output[ct, hour]
            for ct in m.set_country_tech_output
            if ct[0] == country
        )
        - pyo.quicksum(
            m.var_tech_input[ct, hour]
            for ct in m.set_country_tech_input
            if ct[0] == country
        )
        == demand_profile.loc[hour, country]
    )

model_dense.constraint_energy_balance = pyo.Constraint(
    model_dense.set_country,
    model_dense.set_hour,
    rule=energy_balance_rule,
)

print(f"✅ Energy balance in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# -------- Transmission capacity --------
def transmission_capacity_rule(m, line, hour):
    return (
        m.var_transmission[line, hour]
        <= df_line_capacity.loc[line, "Capacity"]
    )

model_dense.constraint_transmission_capacity = pyo.Constraint(
    model_dense.set_line,
    model_dense.set_hour,
    rule=transmission_capacity_rule,
)

print(f"✅ Transmission capacity in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# -------- Tech output capacity --------
def tech_output_capacity_rule(m, country, tech, hour):
    if tech in ["Wind_Onshore", "Wind_Offshore", "Solar", "Run_of_River"]:
        return (
            m.var_tech_output[country, tech, hour]
            <= dict_tech_profile[tech].loc[hour, country]
        )
    elif tech in [
        "Battery", "Nuclear", "Coal", "CCGT", "OCGT", "Hydro_Storage"
    ]:
        return (
            m.var_tech_output[country, tech, hour]
            <= df_tech_capacity.loc[country, tech]
        )
    elif tech == "E_ENS":
        return (
            m.var_tech_output[country, tech, hour]
            <= demand_profile[country].max()
        )
    else:
        raise ValueError(f"Unknown tech: {tech}")

model_dense.constraint_tech_output_capacity = pyo.Constraint(
    model_dense.set_country_tech_output,
    model_dense.set_hour,
    rule=tech_output_capacity_rule,
)

print(f"✅ Tech output capacity in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# -------- Tech input capacity --------
def tech_input_capacity_rule(m, country, tech, hour):
    return (
        m.var_tech_input[country, tech, hour]
        <= df_tech_capacity.loc[country, tech]
    )

model_dense.constraint_tech_input_capacity = pyo.Constraint(
    model_dense.set_country_tech_input,
    model_dense.set_hour,
    rule=tech_input_capacity_rule,
)

print(f"✅ Tech input capacity in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# -------- Storage energy capacity --------
def stored_energy_capacity_rule(m, country, tech, hour):
    if tech == "Battery":
        cap = df_tech_capacity.loc[country, "Battery_MWh"]
    else:
        cap = 1000 * df_tech_capacity.loc[country, f"{tech}_GWh"]
    return m.var_stored_energy[country, tech, hour] <= cap

model_dense.constraint_stored_energy_capacity = pyo.Constraint(
    model_dense.set_country_tech_input,
    model_dense.set_hour,
    rule=stored_energy_capacity_rule,
)

print(f"✅ Storage capacity in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# -------- Storage balance (with self‑dissipation) --------
def storage_balance_rule(m, country, tech, hour):
    eff = dict_storage_cycle_efficiency[tech]

    if WITH_STORAGE_DISSIPATION:
        diss = dict_storage_hourly_self_dissipate_rate[tech]
    else:
        diss = 0.0

    if hour == 1:
        return (
            m.var_stored_energy[country, tech, hour]
            - (1 - diss) * m.var_stored_energy[country, tech, 8760]
            == m.var_tech_input[country, tech, hour]
            - (1 / eff) * m.var_tech_output[country, tech, hour]
        )
    else:
        return (
            m.var_stored_energy[country, tech, hour]
            - (1 - diss) * m.var_stored_energy[country, tech, hour - 1]
            == m.var_tech_input[country, tech, hour]
            - (1 / eff) * m.var_tech_output[country, tech, hour]
        )

model_dense.constraint_storage_balance = pyo.Constraint(
    model_dense.set_country_tech_input,
    model_dense.set_hour,
    rule=storage_balance_rule,
)

print(f"✅ Storage balance in {round(timeit.default_timer() - start_time)}s")
start_time = timeit.default_timer()

# ============================================================
# Objective Function
# ============================================================
def total_cost_rule(m):
    # Transmission tariff cost
    if WITH_GRID_TARIFF:
        trans_cost = pyo.quicksum(
            df_grid_tarif.loc[line].values[0]
            * m.var_transmission[line, hour]
            for line in m.set_line
            for hour in m.set_hour
        )
    else:
        trans_cost = 0

    # Generation variable cost
    gen_cost = pyo.quicksum(
        dict_variable_cost[ct[1]] * m.var_tech_output[ct, hour]
        for ct in m.set_country_tech_output
        for hour in m.set_hour
    )

    return trans_cost + gen_cost

model_dense.obj = pyo.Objective(
    rule=total_cost_rule, sense=pyo.minimize
)

print(f"✅ Objective created in {round(timeit.default_timer() - start_time)}s")

# ============================================================
# Model Summary
# ============================================================
print("\n📊 Dense Model Summary")
print(f"   Variables: {model_dense.nvariables():,}")
print(f"   Constraints: {model_dense.nconstraints():,}")

In [ ]:
# ============================================================
# Solve Dense Model
# ============================================================

# -------- Solver setup --------
solver = pyo.SolverFactory("gurobi")

# Gurobi options (left empty to use defaults)
# Common options for reproducibility & diagnostics:
#
# "Method": 2,          # Barrier (interior point)
# "Presolve": 2,        # Aggressive presolve
# "Seed": 1,            # Deterministic behavior

gurobi_opts = {}

# -------- Log file naming --------
logfile = (
    f"logs/dense_"
    f"{YEAR}_"
    f"(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)}"
    f"_gt{int(WITH_GRID_TARIFF)}_sd{int(WITH_STORAGE_DISSIPATION)}).log"
)

print(f"📝 Log file: {logfile}")

# -------- Solve --------
results, metrics = solve_with_gurobi_logging(
    pyomo_model=model_dense,
    solver=solver,
    log_path=logfile,
    options=gurobi_opts,
)

# ============================================================
# Extract Key Results
# ============================================================

# Objective value
obj_value = pyo.value(model_dense.obj)
metrics["objective_value"] = obj_value

print("\n✅ Dense Model Solved Successfully")
print(f"   Objective value: {obj_value:,.2f}")

# ============================================================
# Save Metrics
# ============================================================

# Prepare metrics dataframe
df_metrics = pd.DataFrame([{
    "model": "dense",
    "year": YEAR,
    "storage": WITH_STORAGE,
    "transmission": WITH_TRANSMISSION,
    "grid_tariff": WITH_GRID_TARIFF,
    "storage_dissipation": WITH_STORAGE_DISSIPATION,
    **metrics
}])

# Save path
save_dir = "results"
os.makedirs(save_dir, exist_ok=True)

save_path = (
    f"{save_dir}/dense_metrics_"
    f"{YEAR}_"
    f"(s{int(WITH_STORAGE)}_t{int(WITH_TRANSMISSION)}"
    f"_gt{int(WITH_GRID_TARIFF)}_sd{int(WITH_STORAGE_DISSIPATION)}).csv"
)

df_metrics.to_csv(save_path, index=False)

print(f"💾 Metrics saved to: {save_path}")

# ============================================================
# Quick Diagnostics
# ============================================================

print("\n📊 Solver Diagnostics")
print(f"   Wall time (Python): {metrics['solver_wall_time_reported']:.2f}s")
print(f"   Memory delta: {metrics['rss_delta_GB']:.2f} GB")